# From a venue's own lines to a book

`tasks/parse_market/parse_market.yml` **is** the job. This notebook runs it, and
then looks at what it did.

The shape of it: a capture of FIX logs read once, every message translated into
the orders and executions it carries, and those folded — per instrument, in time
order — into the book they describe. Three Iceberg tables come out:
`market.orders`, `market.executions` and `market.books`.

Nothing below writes a schema, a partition spec or a sort order. Each table is
created from the declaration of the shape it holds, which is the same
declaration published under `schemas/rekep/`.

## A capture to read

The job in the YAML points at `../data/capture`, which is not in this repository
— a capture is somebody's data, not ours. So this cell writes a small one into a
scratch folder and points the job at that instead. Everything below is exactly
what a real run does.

The messages are a market-data feed: a snapshot, then incremental updates that
add, change, delete and trade against the levels.

In [ ]:
import pathlib
import tempfile

work = pathlib.Path(tempfile.mkdtemp(prefix="parse-market-"))
capture = work / "capture"
capture.mkdir()

#: A log line's own header, which is what the parser reads `runix` off.
HEADER = "2026-08-21 10:30:{second:02d}.000_000 [1-a:b:c] [Bridge] (INFO) "

#: The instrument every message below is about. One capture, one instrument
#: here -- a real one carries thousands, and the job folds each on its own.
ABOUT = "55=BTC-USD|207=XCME|15=USD"


def refresh(second: int, entries: str) -> str:
    """One MarketDataIncrementalRefresh <X>, with `entries` inside it."""
    stamp = f"20260821-10:30:{second:02d}.000"
    return (
        HEADER.format(second=second)
        + f"8=FIX.4.4|35=X|49=XCME|52={stamp}|{ABOUT}|" + entries + "|10=001"
    )


def entry(
    kind: str, action: str, px: float, qty: float, second: int, named: str = "", millis: int = 0
) -> str:
    """One `NoMDEntries <268>` entry: a bid, an offer, or a trade."""
    at = f"272=20260821|273=10:30:{second:02d}.{millis:03d}"
    named = f"|278={named}" if named else ""
    return f"279={action}|269={kind}|270={px}|271={qty}{named}|{at}"


lines = [
    # A snapshot: two levels a side. Each entry carries its own instant, a
    # millisecond apart -- which is what the message's `SendingTime <52>`
    # cannot tell you and `MDEntryTime <273>` can.
    refresh(0, "268=4|" + "|".join([
        entry("0", "0", 100.0, 5, 0, "B1", millis=1),
        entry("0", "0", 99.5, 3, 0, "B2", millis=2),
        entry("1", "0", 100.5, 7, 0, "A1", millis=3),
        entry("1", "0", 101.0, 4, 0, "A2", millis=4),
    ])),
    # More size joins the touch of the bid.
    refresh(1, "268=1|" + entry("0", "1", 100.0, 9, 1, "B1")),
    # A trade prints at the offer, and takes some of it.
    refresh(2, "268=1|" + entry("2", "0", 100.5, 3, 2)),
    # The best offer is pulled entirely.
    refresh(3, "268=1|" + entry("1", "2", 100.5, 0, 3, "A1")),
    # And a new one arrives inside the old spread.
    refresh(4, "268=1|" + entry("1", "0", 100.2, 6, 4, "A3")),
]
(capture / "bridge.log").write_text("\n".join(lines) + "\n")
print(f"{len(lines)} messages in {capture}")
print(lines[1][:120], "...")

## The job, read from the file

`Task.from_yaml` reads the document and dispatches on its `kind` — the same
`from_yaml` that reads a schema contract. Nothing here knows what a
`ParseMarket` is until the file says so.

The two overrides are the scratch capture and a scratch catalog; every other
field is what the file says.

In [ ]:
from rekep import Task

task = Task.from_yaml("parse_market.yml")
warehouse = work / "warehouse"
warehouse.mkdir()
task.source = str(capture)
task.pattern = "*.log"
task.timezone = None                       # the sample header is written in UTC
task.properties = {
    "type": "sql",
    "uri": f"sqlite:///{(work / 'catalog.db').as_posix()}",
    "warehouse": warehouse.as_uri(),
}
task.kind, task.namespace, task.books

## What one message becomes

Before running the whole thing, one line on its own. `FixEvents` is the
translation, and iterating it gives the market events the message carries — here
one per market-data entry, each with **its own** `MDEntryTime <273>` rather than
the message's `SendingTime <52>`.

In [ ]:
from rekep import FixEvents
from rekep.market.fix import unix_of

for event in FixEvents.from_text(lines[0], venue="XCME"):
    print(
        f"{type(event).__name__:10} unix={event.unix} {event.side.name:8}"
        f" px={event.px:<7} qty={event.qty:<5} {event.state.name}"
    )
print()
print("SendingTime <52> was", unix_of("20260821-10:30:00.000"), "-- and is `runix`, not `unix`")

## Running it

One pass over the capture for the events, then one fold per instrument for the
books. The report says what was read, what landed where, and what was already
stored — returned rather than printed, so a scheduler can check it.

In [ ]:
report = task.run()
print(report)
report.written

## The book it folded

One row per instant that changed the book, never one per message. Every price
across the two sides is a **column**, computed once at write time: a reader
never recomputes a mid, and an engine can prune on one.

In [ ]:
books = task.target("books").read_arrow_table().sort_by("unix")
books.select(
    ["unix", "bid_px", "bid_qty", "ask_px", "ask_qty", "px", "spread", "micro_px", "imbalance"]
).to_pylist()

Read down the `spread` column and the capture is legible without opening it.

The first four rows are the snapshot arriving: each of its entries carries its
own `MDEntryTime <273>`, a millisecond apart, so the book is four states and not
one — and there is no mid at all until an offer shows up, rather than a mid
computed against nothing. Then the bid's size grows to 9, the trade at 10:30:02
takes three off the offer without moving it, the pull at 10:30:03 widens the
spread to 1.0, and the new offer inside it narrows it to 0.2.

## The levels under it

`bid_alive` and `ask_alive` are the levels themselves, best first, with the
number of orders making up each one — which is the one thing an aggregated feed
cannot tell you and an order-by-order fold can.

In [ ]:
#: The fourth row: the last entry of the snapshot, so the whole of it is applied.
row = books.slice(3, 1).to_pylist()[0]
print("bid:", [(level["px"], level["qty"], level["orders"]) for level in row["bid_alive"]])
print("ask:", [(level["px"], level["qty"], level["orders"]) for level in row["ask_alive"]])
print()
last = books.slice(books.num_rows - 1, 1).to_pylist()[0]
print("at the end -- ask:", [(one["px"], one["qty"]) for one in last["ask_alive"]])

## The events it was folded from

Both are tables of their own, because they answer different questions: an order
is what somebody asked for, an execution is what actually moved. Storing only
the book would lose both.

In [ ]:
orders = task.target("orders").read_arrow_table().sort_by("unix")
orders.select(["unix", "side", "px", "qty", "state", "order_id", "instrument_hash"]).slice(0, 6).to_pylist()

In [ ]:
fills = task.target("executions").read_arrow_table()
print(f"{fills.num_rows} executions")
fills.select(["unix", "px", "qty", "kind", "state"]).to_pylist()

## How the tables are laid out

Nothing above chose this: it is read off the declaration when each table is
created. `unix_hour` is the hour by identity, `instrument_hash` is a sixteen-way
bucket — one instrument's stream lands in one of sixteen files an hour rather
than in a directory of its own — and the rows inside are sorted by `unix`.

Which is exactly the layout `Book.from_events` wants to read back: one
instrument, in time order.

In [ ]:
from rekep.market import Book, Order

for shape in ("orders", "books"):
    declared = task.target_field(shape)
    print(f"{shape:8} partition={declared.partition_keys()} sort={declared.sort_keys()}")
print()
print("identifiers are int64 everywhere:", Order.FIELD.field("hash").arrow_type)
print("a book's own identity:", books.column("xhash").to_pylist()[0])
print("versions:", books.column("version").to_pylist())

## Running it again

The point of appending with `merge_by`: a replay reads the capture and writes
nothing. Nothing stored is rewritten, and no delete file is produced.

In [ ]:
print(task.run())

## Chaining it onto `parse_logs`

The source above is a folder of text. It can equally be a **dataset document**,
which is how this reads the output of `parse_logs` instead of the capture:
`kind` names the store and `Dataset.from_dict` finds the class for it.

Nothing else about the job changes — the same translation, the same fold, the
same three tables.

In [ ]:
task.source = {
    "kind": "iceberg",
    "name": "logs.book_side_logs",      # what `parse_logs` lands FIX refreshes in
    "catalog": task.catalog,
    "properties": task.properties,
}
reading = task.source_dataset()
print(type(reading).__name__, "->", reading.name, "-- reading column", task.column)

## Cleaning up

In [ ]:
import shutil

shutil.rmtree(work, ignore_errors=True)
print("gone:", work)